# Unity Catalog

A brief overview of the main Unity Catalog features, with syntax and small examples.

## 1. What Unity Catalog gives you

Unity Catalog is Databricks' governance layer for data, AI and other securables. It gives you:
* a three-level namespace: **catalog.schema.object**
* centralized permissions
* data lineage and auditability
* data discovery with comments and tags
* governance features such as PII classification, ABAC policies, row filters and column masks

## 2. Three-level namespace

| Level | Example | Purpose |
|---|---|---|
| Catalog | `training` | top-level container, often by domain or environment |
| Schema | `training.bronze` | logical grouping inside a catalog |
| Object | `training.bronze.customers` | table, view, volume, function, model, etc. |

### Syntax
```sql
CREATE CATALOG IF NOT EXISTS training;
CREATE SCHEMA IF NOT EXISTS training.bronze;
CREATE TABLE IF NOT EXISTS training.bronze.customers (
  customer_id BIGINT,
  customer_name STRING
);
```

## 3. Hands-on — permissions step by step

The trainer removes all catalog access of `one_developers`, then grants one privilege per level. After each level a trainee re-runs the query to see what is still missing.

| Level | Trainer grants | Trainee query on `customers` |
|---|---|---|
| 0 | nothing | fails |
| 1 | `SELECT` on the table | fails — `USE CATALOG` missing |
| 2 | + `USE CATALOG` | fails — `USE SCHEMA` missing |
| 3 | + `USE SCHEMA` | works (but `orders` still fails, no `SELECT`) |

The syntax and grants per role are in **section 4**. The original grants are restored in **End** (below) or in **section 14**.

In [0]:
# Save current catalog grants of one_developers (needed for restore at the end)
spark.sql("""
  CREATE TABLE IF NOT EXISTS training.default.one_developers_catalog_grants_backup AS
  SELECT catalog_name, grantee, privilege_type, current_timestamp() AS backup_ts
  FROM system.information_schema.catalog_privileges
  WHERE catalog_name = 'training' AND grantee = 'one_developers' AND inherited_from = 'NONE'
""")
display(spark.table("training.default.one_developers_catalog_grants_backup"))

In [0]:
%sql
-- Remove all catalog-level access for the group
REVOKE ALL PRIVILEGES ON CATALOG training FROM `one_developers`;

-- demo schema with two tables
CREATE SCHEMA IF NOT EXISTS training.uc_demo;
CREATE OR REPLACE TABLE training.uc_demo.customers AS SELECT * FROM samples.tpch.customer LIMIT 100;
CREATE OR REPLACE TABLE training.uc_demo.orders    AS SELECT * FROM samples.tpch.orders   LIMIT 100;

SHOW GRANTS `one_developers` ON CATALOG training;   -- expect 0 rows

In [0]:
# Run before each level to start from zero
def reset_demo_grants(group="one_developers"):
    for stmt in [
        f"REVOKE ALL PRIVILEGES ON CATALOG training FROM `{group}`",
        f"REVOKE ALL PRIVILEGES ON SCHEMA training.uc_demo FROM `{group}`",
        f"REVOKE MANAGE ON SCHEMA training.uc_demo FROM `{group}`",
    ] + [
        s for t in ["customers", "orders"] for s in (
            f"REVOKE ALL PRIVILEGES ON TABLE training.uc_demo.{t} FROM `{group}`",
            f"REVOKE MANAGE ON TABLE training.uc_demo.{t} FROM `{group}`",
        )
    ]:
        spark.sql(stmt)
    print("Demo grants cleared for", group)

reset_demo_grants()

In [0]:
%sql

SELECT * FROM training.uc_demo.customers LIMIT 5;

In [0]:
%sql
-- Table access only. Trainee re-runs the query -> fails, USE CATALOG missing
GRANT SELECT ON TABLE training.uc_demo.customers TO `one_developers`;

In [0]:
%sql
-- Trainee re-runs -> still fails, USE SCHEMA missing
GRANT USE CATALOG ON CATALOG training TO `one_developers`;

In [0]:
%sql create table training.uc_demo.customers_3 as select * from training.uc_demo.customers  

In [0]:
%sql
-- Trainee re-runs -> works now
GRANT USE SCHEMA ON SCHEMA training.uc_demo TO `one_developers`;

SHOW GRANTS `one_developers` ON TABLE training.uc_demo.customers;

In [0]:
%sql
-- TRAINEE: fails, no SELECT on orders
SELECT * FROM training.uc_demo.orders LIMIT 5;

In [0]:
%sql GRANT select ON SCHEMA training.uc_demo TO `one_developers`;

In [0]:
%sql GRANT select ON catalog training TO `one_developers`;

In [0]:
# Remove demo objects and restore the original catalog grants
reset_demo_grants()
spark.sql("DROP SCHEMA IF EXISTS training.uc_demo CASCADE")

backup = spark.table("training.default.one_developers_catalog_grants_backup").collect()
for r in backup:
    priv = r.privilege_type.replace("_", " ")
    spark.sql(f"GRANT {priv} ON CATALOG training TO `{r.grantee}`")
print(f"Restored {len(backup)} privileges")

display(spark.sql("SHOW GRANTS `one_developers` ON CATALOG training"))  # compare count with the backup
# when the count matches: spark.sql("DROP TABLE training.default.one_developers_catalog_grants_backup")

## 4. Permissions and grants

Unity Catalog controls access with `GRANT` / `REVOKE`.

### Syntax
```sql
GRANT USE CATALOG ON CATALOG training TO `account users`;
GRANT USE SCHEMA ON SCHEMA training.bronze TO `account users`;
GRANT SELECT ON TABLE training.bronze.customers TO `one_developers`;
REVOKE SELECT ON TABLE training.bronze.customers FROM `one_developers`;
```

### Check permissions
```sql
SHOW GRANTS ON TABLE training.bronze.customers;
```

### Grants by role
Assumed: developer `developer@one-consultants.com`, admin `admin@one-consultants.com`, prod service principal `<sp-app-id>`, catalogs `training_dev` / `training_prod`. Replace `<...>` with real IDs.
Unity Catalog = SQL `GRANT`. Jobs, pipelines, folders, clusters, warehouses = Databricks CLI `update-permissions`.

#### DEV - Developer
```sql
GRANT USE CATALOG, BROWSE, CREATE SCHEMA, USE SCHEMA, SELECT, MODIFY,
      CREATE TABLE, CREATE MATERIALIZED VIEW, CREATE FUNCTION, CREATE VOLUME,
      READ VOLUME, WRITE VOLUME, EXECUTE, APPLY TAG
  ON CATALOG training_dev TO `developer@one-consultants.com`;
GRANT READ FILES ON EXTERNAL LOCATION inboundfiles_location TO `developer@one-consultants.com`;
```
```bash
databricks jobs      update-permissions <dev_job_id>      --json '{"access_control_list":[{"user_name":"developer@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
databricks pipelines update-permissions <dev_pipeline_id> --json '{"access_control_list":[{"user_name":"developer@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
databricks workspace update-permissions directories <dev_folder_id> --json '{"access_control_list":[{"user_name":"developer@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
databricks clusters  update-permissions <dev_cluster_id>  --json '{"access_control_list":[{"user_name":"developer@one-consultants.com","permission_level":"CAN_RESTART"}]}'
databricks warehouses update-permissions <dev_warehouse_id> --json '{"access_control_list":[{"user_name":"developer@one-consultants.com","permission_level":"CAN_USE"}]}'
```

#### DEV - Admin
```sql
GRANT CREATE CATALOG, CREATE EXTERNAL LOCATION, CREATE STORAGE CREDENTIAL ON METASTORE TO `admin@one-consultants.com`;
ALTER CATALOG training_dev OWNER TO `admin@one-consultants.com`;
GRANT ALL PRIVILEGES ON EXTERNAL LOCATION inboundfiles_location TO `admin@one-consultants.com`;
```
```bash
databricks jobs      update-permissions <dev_job_id>      --json '{"access_control_list":[{"user_name":"admin@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
databricks pipelines update-permissions <dev_pipeline_id> --json '{"access_control_list":[{"user_name":"admin@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
databricks workspace update-permissions directories <dev_folder_id> --json '{"access_control_list":[{"user_name":"admin@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
databricks clusters  update-permissions <dev_cluster_id>  --json '{"access_control_list":[{"user_name":"admin@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
databricks warehouses update-permissions <dev_warehouse_id> --json '{"access_control_list":[{"user_name":"admin@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
```

#### PROD - Service principal (deploys and runs jobs / pipelines)
```sql
GRANT USE CATALOG, USE SCHEMA, SELECT, MODIFY,
      CREATE TABLE, CREATE MATERIALIZED VIEW, CREATE FUNCTION
  ON CATALOG training_prod TO `<sp-app-id>`;
GRANT READ FILES ON EXTERNAL LOCATION inboundfiles_prod_location TO `<sp-app-id>`;
```
```bash
databricks jobs      update-permissions <prod_job_id>      --json '{"access_control_list":[{"service_principal_name":"<sp-app-id>","permission_level":"IS_OWNER"}]}'
databricks pipelines update-permissions <prod_pipeline_id> --json '{"access_control_list":[{"service_principal_name":"<sp-app-id>","permission_level":"IS_OWNER"}]}'
databricks workspace update-permissions directories <prod_folder_id> --json '{"access_control_list":[{"service_principal_name":"<sp-app-id>","permission_level":"CAN_MANAGE"}]}'
```

#### PROD - Developer (read only)
```sql
GRANT USE CATALOG, BROWSE, USE SCHEMA, SELECT ON CATALOG training_prod TO `developer@one-consultants.com`;
```
```bash
databricks jobs      update-permissions <prod_job_id>      --json '{"access_control_list":[{"user_name":"developer@one-consultants.com","permission_level":"CAN_VIEW"}]}'
databricks pipelines update-permissions <prod_pipeline_id> --json '{"access_control_list":[{"user_name":"developer@one-consultants.com","permission_level":"CAN_VIEW"}]}'
databricks workspace update-permissions directories <prod_folder_id> --json '{"access_control_list":[{"user_name":"developer@one-consultants.com","permission_level":"CAN_READ"}]}'
databricks warehouses update-permissions <prod_warehouse_id> --json '{"access_control_list":[{"user_name":"developer@one-consultants.com","permission_level":"CAN_USE"}]}'
```

#### PROD - Admin
```sql
ALTER CATALOG training_prod OWNER TO `admin@one-consultants.com`;
GRANT ALL PRIVILEGES ON EXTERNAL LOCATION inboundfiles_prod_location TO `admin@one-consultants.com`;
```
```bash
databricks jobs      update-permissions <prod_job_id>      --json '{"access_control_list":[{"user_name":"admin@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
databricks pipelines update-permissions <prod_pipeline_id> --json '{"access_control_list":[{"user_name":"admin@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
databricks workspace update-permissions directories <prod_folder_id> --json '{"access_control_list":[{"user_name":"admin@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
databricks warehouses update-permissions <prod_warehouse_id> --json '{"access_control_list":[{"user_name":"admin@one-consultants.com","permission_level":"CAN_MANAGE"}]}'
```

#### Verify / revoke
```sql
SHOW GRANTS `developer@one-consultants.com` ON CATALOG training_prod;
REVOKE MODIFY ON CATALOG training_prod FROM `developer@one-consultants.com`;
```
```bash
databricks jobs get-permissions <prod_job_id>
```

## 5. Views and functions

### View syntax
```sql
CREATE OR REPLACE VIEW training.silver.active_customers AS
SELECT *
FROM training.bronze.customers
WHERE customer_name IS NOT NULL;
```

### Function syntax
```sql
CREATE OR REPLACE FUNCTION training.default.mask_email(email STRING)
RETURNS STRING
RETURN regexp_replace(email, '(^.).*(@.*$)', '$1***$2');
```

Views help standardize logic. Functions help reuse logic, especially for governance and masking.

## 6. Comments for documentation

Comments improve discoverability in Catalog Explorer.

### Syntax
```sql
COMMENT ON TABLE training.bronze.customers IS 'Raw customer records from the source system';
COMMENT ON COLUMN training.bronze.customers.customer_name IS 'Customer full name';
```

## 7. Tags

Tags attach metadata to catalogs, schemas, tables and columns.

### Table tag syntax
```sql
SET TAG ON TABLE training.bronze.customers domain = customer;
SET TAG ON TABLE training.bronze.customers quality_tier = bronze;
```

### Column tag syntax
```sql
SET TAG ON COLUMN training.bronze.customers.customer_name pii = true;
SET TAG ON COLUMN training.bronze.customers.email pii_type = email;
```

### Query tags
```sql
SELECT catalog_name, schema_name, table_name, column_name, tag_name, tag_value
FROM system.information_schema.column_tags
WHERE tag_name = 'pii';
```

Tags are often the bridge between discovery and policy enforcement.

## 8. PII and data classification

Unity Catalog can automatically classify sensitive data such as PII, financial data and credentials, and store the results for governance workflows. Databricks data classification uses an agent to scan tables and can auto-tag sensitive columns, and the results are queryable in `system.data_classification.results`. [Data Classification](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/data-classification/) [Data classification system table reference](https://learn.microsoft.com/en-us/azure/databricks/admin/system-tables/data-classification/)

### Enable it first
`system.data_classification.results` does **not exist until Data Classification is enabled** on at least one catalog.
1. Catalog Explorer > open the catalog (e.g. `training`) > **Details** tab
2. Next to **Data Classification**, click **Enable**, choose the schemas, click **Save**
3. A background job scans the tables (new tables are usually scanned within 24 hours)

Requirements:
* catalog owner or `MANAGE` on the catalog
* serverless compute (the results table can only be queried from serverless)
* by default only **account admins** can read the results table; they can grant access to others

### Example query (after enabling)
```sql
SELECT *
FROM system.data_classification.results
WHERE catalog_name = 'training'
  AND confidence = 'HIGH';
```

### Fallback: tag PII manually and query the tags
If classification is not enabled, tag PII columns yourself and find them with `information_schema`:
```sql
SET TAG ON COLUMN training.silver.customers.email pii_type = email;

SELECT catalog_name, schema_name, table_name, column_name, tag_value AS pii_type
FROM system.information_schema.column_tags
WHERE tag_name = 'pii_type';
```

This helps you find columns that may need masking or row-level protection.

In [0]:
%sql
SELECT *
FROM system.data_classification.results
WHERE catalog_name = 'training'

## 9. Row filters

A row filter limits which rows a user can see.

### Manual row filter on one table
```sql
ALTER TABLE training.silver.orders
SET ROW FILTER training.default.us_only_filter ON (country);
```

The function must return `BOOLEAN`: `TRUE` keeps the row.

### Example row filter function
```sql
CREATE OR REPLACE FUNCTION training.default.us_only_filter(country STRING)
RETURNS BOOLEAN
RETURN country = 'USA';
```

Use row filters when the same table should show different rows to different users.

## 10. Column masks

A column mask changes the value a user sees in a column.

### Manual mask on one table
```sql
ALTER TABLE training.silver.customers
ALTER COLUMN email
SET MASK training.default.mask_email;
```

### Example masking function
```sql
CREATE OR REPLACE FUNCTION training.default.mask_phone(phone STRING)
RETURNS STRING
RETURN concat('***-***-', right(phone, 4));
```

Column masks are useful for email, phone, SSN, salary and similar fields.

## 11. ABAC policies

ABAC means **attribute-based access control**. In Unity Catalog, ABAC policies use **governed tags** to dynamically apply row filters or column masks across many tables. Governed tags are the recommended tag type for ABAC policies. `CREATE POLICY` is the policy syntax; it is different from a manual `ALTER TABLE ... SET ROW FILTER` or `SET MASK` on one table. If enabled in your workspace, ABAC policies can be attached at metastore, catalog, schema or table scope and match tables/columns by governed tags. [Create and manage policies](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/policies) [Requirements, quotas, and limitations](https://docs.databricks.com/aws/en/data-governance/unity-catalog/abac/requirements)

### Example governed-tag column mask policy for PII email
```sql
CREATE POLICY email_redaction
ON SCHEMA training.silver
COLUMN MASK training.default.mask_email
TO `account users` EXCEPT `balaji.raghu@one-consultants.com`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_type', 'email') AS email
ON COLUMN email;
```

### Example row filter policy
```sql
CREATE POLICY region_filter
ON SCHEMA training.silver
ROW FILTER training.default.us_only_filter
TO `one_developers`
FOR TABLES
MATCH COLUMNS has_tag('geo_region') AS region
USING COLUMNS (region);
```

### Check policies
```sql
SHOW POLICIES ON SCHEMA training.silver;
SHOW EFFECTIVE POLICIES ON TABLE training.silver.customers;
```

**Difference from manual filters and masks:**
* manual row filters / masks apply to one table
* ABAC policies apply automatically to any table or column whose governed tags match the policy

### Step-by-step: ABAC row filter policy

**Goal:** members of `one_developers` only see customers from the `AMERICA` region in any table of `training.silver` whose region column is tagged `geo_region`. Everyone else sees all rows.

**Principals used** (from this workspace)

| Principal | Type | Role in the example |
|---|---|---|
| `one_developers` | account group (11 members) | filtered by the policy (`TO`) |
| `balaji.raghu@one-consultants.com` | user (trainer) | exempt from the policy (`EXCEPT`) |
| `account users` | built-in account group | everyone in the account |

> `admins` and `users` are **workspace-local** groups and cannot be used in Unity Catalog grants or policies. Use account groups (or `account users`) instead.

**Prerequisites**
* Compute: serverless, or standard / dedicated compute on **Databricks Runtime 16.4+** (older runtimes are denied access to ABAC-secured tables)
* Permission to create governed tags (account admin, or `CREATE` / `MANAGE` on governed tags) and `ASSIGN` on the tags
* `MANAGE` on the schema `training.silver` (to create the policy) and `APPLY TAG` on the table
* `CREATE GOVERNED TAG` in SQL needs Databricks SQL or Runtime 18.1+ (or use the UI: **Catalog > Govern > Governed Tags**)

> Run the steps below **one cell at a time, in order**.

#### Step 1 — Create a sample table with a region column
Builds `training.silver.customer_abac` from `samples.tpch` and gives `one_developers` read access.

In [0]:
%sql
CREATE OR REPLACE TABLE training.silver.customer_abac AS
SELECT c.c_custkey, c.c_name, c.c_mktsegment, r.r_name AS region
FROM samples.tpch.customer c
JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey
JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey;

GRANT SELECT ON TABLE training.silver.customer_abac TO `one_developers`;

In [0]:
%sql
-- Expected: 5 regions, ~150,000 customers each
SELECT region, count(*) AS customers
FROM training.silver.customer_abac
GROUP BY region
ORDER BY region;

- #### Step 2 — Create the governed tag `geo_region` (once per account)
Needs governed-tag create permission and Databricks SQL / Runtime 18.1+ (serverless works). If the tag already exists, skip this cell. Alternative: **Catalog > Govern > Governed Tags** in the UI.

In [0]:
%sql
-- key-only tag (no values)
CREATE GOVERNED TAG geo_region
  DESCRIPTION 'Marks the column that holds the customer region';

In [0]:
%sql
SHOW GOVERNED TAGS;

#### Step 3 — Tag the region column
Column tag `geo_region` → used by the policy's `MATCH COLUMNS` to find the column.

In [0]:
%sql
SET TAG ON COLUMN training.silver.customer_abac.region geo_region;

#### Step 4 — Check the tags
Expect 1 row: the `geo_region` tag on column `region`.
> Tag changes can take a few minutes to take effect.

In [0]:
%sql

SELECT 'column', table_name, column_name, tag_name, tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = 'training' AND schema_name = 'silver' AND table_name = 'customer_abac';

#### Step 5 — Create the row filter function
The function returns `TRUE` for rows to keep. Keep **who** it applies to out of the function — that is handled by the policy's `TO` / `EXCEPT`.

In [0]:
%sql
CREATE OR REPLACE FUNCTION training.default.filter_america(region STRING)
RETURNS BOOLEAN
RETURN region = 'AMERICA';

#### Step 6 — Create the ABAC row filter policy

| Clause | Meaning |
|---|---|
| `ON SCHEMA` | scope: every table in `training.silver` (inherited) |
| `ROW FILTER` | the boolean function to apply |
| `TO` / `EXCEPT` | who is filtered / who is exempt |
| `MATCH COLUMNS ... AS region` | finds the column tagged `geo_region` |
| `USING COLUMNS (region)` | passes that column to the function |

In [0]:
%sql
CREATE POLICY america_only
ON SCHEMA training.silver
COMMENT 'one_developers only see AMERICA rows on tables with a geo_region column'
ROW FILTER training.default.filter_america
TO `one_developers`
EXCEPT `balaji.raghu@one-consultants.com`        -- exempt trusted users / ETL service principals
FOR TABLES
MATCH COLUMNS has_tag('geo_region') AS region    -- which column to pass to the function
USING COLUMNS (region);

#### Step 7 — Verify the policy
Three checks: policies on the schema, policies that apply to the table (incl. inherited), and the full policy definition.

In [0]:
%sql
SHOW POLICIES ON SCHEMA training.silver;

In [0]:
%sql
SHOW EFFECTIVE POLICIES ON TABLE training.silver.customer_abac;

In [0]:
%sql
DESCRIBE POLICY america_only ON SCHEMA training.silver;

#### Step 8 — Test as different users
Run the next cell yourself, then ask a member of `one_developers` (e.g. Philippe Cooman, Sravan Koutharapu) to run it.

| Who runs the query | Result |
|---|---|
| member of `one_developers` | 1 row: `AMERICA` (~150,000) |
| `balaji.raghu@one-consultants.com` (EXCEPT) or anyone not in `one_developers` | all 5 regions (~750,000) |

In [0]:
%sql
SELECT region, count(*) AS customers
FROM training.silver.customer_abac
GROUP BY region
ORDER BY region;

#### Step 9 — Extend to more tables (no new policy needed)
Create a second table in `training.silver` and tag it the same way — the existing `america_only` policy applies to it automatically.

In [0]:
%sql
CREATE OR REPLACE TABLE training.silver.orders_by_region AS
SELECT r.r_name AS region,
       count(*) AS order_count,
       round(sum(o.o_totalprice), 2) AS total_price
FROM samples.tpch.orders o
JOIN samples.tpch.customer c ON o.o_custkey = c.c_custkey
JOIN samples.tpch.nation n   ON c.c_nationkey = n.n_nationkey
JOIN samples.tpch.region r   ON n.n_regionkey = r.r_regionkey
GROUP BY r.r_name;

GRANT SELECT ON TABLE training.silver.orders_by_region TO `one_developers`;

SET TAG ON COLUMN training.silver.orders_by_region.region geo_region;

In [0]:
%sql
-- one_developers members see only the AMERICA row; you see all 5
SELECT * FROM training.silver.orders_by_region ORDER BY region;

#### Step 10 — Clean up
> Drop the policy **before** deleting a governed tag it uses — otherwise queries in the policy's scope fail.

In [0]:
%sql
DROP POLICY america_only ON SCHEMA training.silver;

UNSET TAG ON COLUMN training.silver.orders_by_region.region geo_region;
UNSET TAG ON COLUMN training.silver.customer_abac.region geo_region;

DROP FUNCTION IF EXISTS training.default.filter_america;
DROP TABLE IF EXISTS training.silver.orders_by_region;
DROP TABLE IF EXISTS training.silver.customer_abac;

-- Optional: remove the governed tag (only if no other policy uses it)
-- DROP GOVERNED TAG geo_region;

## 11a. Hands-on — row filter: table level, then policy level

Replace `john` with your name in **Steps 1, 2, 8** and **Clean up**.

Step 9 needs `ASSIGN` on the governed tag `geo_region` (Catalog > Govern > Governed Tags > Permissions).

* Steps 1-5: row filter attached directly to one table
* Steps 6-9: row filter attached by a policy to every table with a `geo_region` tag

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS training.dev_john;

In [0]:
%sql
USE training.dev_john;

In [0]:
%sql
CREATE OR REPLACE TABLE customers AS
SELECT c.c_custkey, c.c_name, r.r_name AS region
FROM samples.tpch.customer c
JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey
JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey
LIMIT 1000;

In [0]:
%sql
-- TRUE keeps the row
CREATE OR REPLACE FUNCTION filter_europe(region STRING)
RETURNS BOOLEAN
RETURN region = 'EUROPE';

In [0]:
%sql
ALTER TABLE customers SET ROW FILTER filter_europe ON (region);

In [0]:
%sql
-- only EUROPE
SELECT region, count(*) AS rows FROM customers GROUP BY region;

In [0]:
%sql
CREATE OR REPLACE TABLE suppliers AS
SELECT s.s_suppkey, s.s_name, r.r_name AS region
FROM samples.tpch.supplier s
JOIN samples.tpch.nation n ON s.s_nationkey = n.n_nationkey
JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey
LIMIT 1000;

In [0]:
%sql
-- all 5 regions
SELECT region, count(*) AS rows FROM suppliers GROUP BY region;

In [0]:
%sql
CREATE OR REPLACE POLICY europe_only
ON SCHEMA training.dev_john
ROW FILTER training.dev_john.filter_europe
TO `account users`
FOR TABLES
MATCH COLUMNS has_tag('geo_region') AS region
USING COLUMNS (region);

In [0]:
%sql
SET TAG ON COLUMN suppliers.region geo_region;

In [0]:
%sql
-- only EUROPE now
SELECT region, count(*) AS rows FROM suppliers GROUP BY region;

In [0]:
%sql
DROP SCHEMA IF EXISTS training.dev_john CASCADE;


## 12. Auditing and lineage

Unity Catalog also supports lineage and audit-style analysis through system tables.

### Example lineage query
```sql
SELECT *
FROM system.access.table_lineage
WHERE target_table_full_name = 'training.silver.customers';
```

### Example access history query
```sql
SELECT *
FROM system.access.audit
WHERE service_name = 'unityCatalog';
```

---

## 13. Common governance pattern

A common pattern is:
1. store data in Unity Catalog tables
2. add comments and tags
3. classify sensitive columns
4. apply row filters or masks manually for one table
5. move to ABAC policies when you want the rule to apply dynamically across many tables

### Example sequence
```sql
SET TAG ON COLUMN training.silver.customers.email pii_type = email;

CREATE POLICY email_redaction
ON SCHEMA training.silver
COLUMN MASK training.default.mask_email
TO `account users`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_type', 'email') AS email
ON COLUMN email;
```

---

## 14. Full cleanup — undo everything this notebook did

Run at the very end of the training. Safe to re-run: objects that are already gone are skipped.

| What the notebook did | Section | Undo |
|---|---|---|
| Backed up + revoked `one_developers` catalog grants | 3 (Step 0) | restore grants from the backup, revoke extra demo grants, drop the backup table |
| Schema `training.uc_demo` + tables + grants | 3 | `DROP SCHEMA ... CASCADE` |
| ABAC policy `america_only` on `training.silver` | 11, Step 6 | `DROP POLICY` (**first**, before the tag) |
| Column tags `geo_region` on `customer_abac` / `orders_by_region` | 11, Steps 3 + 9 | `UNSET TAG` |
| Tables `training.silver.customer_abac`, `orders_by_region` + grants | 11, Steps 1 + 9 | `DROP TABLE` |
| Function `training.default.filter_america` | 11, Step 5 | `DROP FUNCTION` |
| Governed tag `geo_region` | 11, Step 2 | `DROP GOVERNED TAG` (only if no column still uses it) |
| Trainee schemas `training.dev_<name>` (tables, `filter_europe`, row filter, policy `europe_only`, tags) | 11a | drop policy, then `DROP SCHEMA ... CASCADE` |
| Current schema set by `USE training.dev_john` | 11a, Step 2 | `USE training.default` |

> Add every trainee schema to `TRAINEE_SCHEMAS` in the next cell before running it.

In [0]:
# Section 14: undo everything this notebook created or changed. Safe to re-run.
GROUP = "one_developers"
TRAINEE_SCHEMAS = ["dev_john"]   # add every training.dev_<name> schema created in section 11a
DROP_GOVERNED_TAG = True          # geo_region was created in section 11, Step 2

def run(stmt):
    try:
        spark.sql(stmt)
        print("OK   ", stmt)
    except Exception as e:
        print("SKIP ", stmt, "->", str(e).splitlines()[0][:150])

# 1. ABAC policies first: a governed tag must not be removed while a policy uses it
run("DROP POLICY IF EXISTS america_only ON SCHEMA training.silver")
for s in TRAINEE_SCHEMAS:
    run(f"DROP POLICY IF EXISTS europe_only ON SCHEMA training.{s}")

# 2. Column tags
for t in ["customer_abac", "orders_by_region"]:
    run(f"UNSET TAG ON COLUMN training.silver.{t}.region geo_region")

# 3. Demo tables (their grants go with them) and the row filter function
for t in ["customer_abac", "orders_by_region"]:
    run(f"DROP TABLE IF EXISTS training.silver.{t}")
run("DROP FUNCTION IF EXISTS training.default.filter_america")

# 4. Demo and trainee schemas (CASCADE drops their tables, functions, row filters, tags and grants)
run("USE training.default")
run("DROP SCHEMA IF EXISTS training.uc_demo CASCADE")
for s in TRAINEE_SCHEMAS:
    run(f"DROP SCHEMA IF EXISTS training.{s} CASCADE")

# 5. Governed tag (only if no column in the metastore still uses it)
if DROP_GOVERNED_TAG:
    in_use = spark.sql(
        "SELECT count(*) AS n FROM system.information_schema.column_tags WHERE tag_name = 'geo_region'"
    ).first().n
    if in_use == 0:
        run("DROP GOVERNED TAG geo_region")
    else:
        print(f"KEEP  governed tag geo_region: still set on {in_use} column(s)")

# 6. Restore the original catalog grants of the group exactly as in the backup
backup_tbl = "training.default.one_developers_catalog_grants_backup"
def current_grants():
    rows = spark.sql(f"SHOW GRANTS `{GROUP}` ON CATALOG training").collect()
    return {r.ActionType.replace(" ", "_") for r in rows if r.Principal == GROUP}

if spark.catalog.tableExists(backup_tbl):
    expected = {r.privilege_type for r in spark.table(backup_tbl).select("privilege_type").distinct().collect()}
    for p in sorted(expected - current_grants()):
        run(f"GRANT {p.replace('_', ' ')} ON CATALOG training TO `{GROUP}`")
    for p in sorted(current_grants() - expected):          # demo grants that were not there before
        run(f"REVOKE {p.replace('_', ' ')} ON CATALOG training FROM `{GROUP}`")
    diff = expected ^ current_grants()
    if not diff:
        run(f"DROP TABLE {backup_tbl}")
        print(f"Restored {len(expected)} catalog privileges for {GROUP}; backup table dropped")
    else:
        print("Grants differ from the backup, backup table KEPT:", sorted(diff))
else:
    print("No backup table found: catalog grants left unchanged")

In [0]:
%sql
-- Section 14 check. Expected: n = 0 on every row
SELECT 'demo / trainee schemas' AS check_name, count(*) AS n
FROM training.information_schema.schemata
WHERE schema_name = 'uc_demo' OR schema_name LIKE 'dev\_%'
UNION ALL
SELECT 'silver demo tables', count(*)
FROM training.information_schema.tables
WHERE table_schema = 'silver' AND table_name IN ('customer_abac', 'orders_by_region')
UNION ALL
SELECT 'filter_america function', count(*)
FROM training.information_schema.routines
WHERE routine_schema = 'default' AND routine_name = 'filter_america'
UNION ALL
SELECT 'geo_region column tags', count(*)
FROM system.information_schema.column_tags
WHERE tag_name = 'geo_region'
UNION ALL
SELECT 'grants backup table', count(*)
FROM training.information_schema.tables
WHERE table_schema = 'default' AND table_name = 'one_developers_catalog_grants_backup';

In [0]:
%sql
-- Section 14 check. Expected: america_only is not listed
SHOW POLICIES ON SCHEMA training.silver;